# Chapter 30 — Your Dissertation in Python

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/ch30_dissertation_in_python.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## 30.1 From a topic to a question

## 30.2 A project folder

In [ ]:
import importlib
import shutil
import sys
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

BOOK = Path.cwd()                               # the book folder
PROJECT = Path(tempfile.gettempdir()) / "salaried_jobs"
shutil.rmtree(PROJECT, ignore_errors=True)      # start this demo from empty
for sub in ["data/raw", "data/clean", "code", "output/tables",
            "output/figures", "paper"]:
    (PROJECT / sub).mkdir(parents=True, exist_ok=True)


def tree(folder: Path, indent: str = "") -> None:
    """Print a folder and its contents, folders first."""
    for item in sorted(folder.iterdir(), key=lambda p: (p.is_file(), p.name)):
        if item.name.startswith("__"):
            continue
        print(indent + item.name + ("/" if item.is_dir() else ""))
        if item.is_dir():
            tree(item, indent + "    ")


tree(PROJECT)

## 30.3 Raw data and a data log

In [ ]:
from indiadata import mospi_nas

mospi_nas(25)                                   # make sure the cached file exists
cached = BOOK / ("data/raw/mospi_nas_getNASData_base_year-2011-12_"
                 "frequency_code-1_indicator_code-25_series-Current.csv")
raw_income = PROJECT / "data/raw/mospi_nas_pc_nsdp_code25.csv"
shutil.copy2(cached, raw_income)                # copy2 keeps the file's dates

PLFS_DIR = BOOK / "data/plfs"
fetched = pd.read_csv(raw_income, usecols=["fetched_on"])["fetched_on"].iloc[0]
plfs_date = pd.Timestamp((PLFS_DIR / "perv1.csv").stat().st_mtime, unit="s")
print("NAS file fetched on:", fetched)
print("PLFS perv1.csv saved on:", plfs_date.date())

In [ ]:
data_log = pd.DataFrame([
    {"file": "data/raw/mospi_nas_pc_nsdp_code25.csv",
     "dataset": "Per capita NSDP by state, current and 2011-12 prices",
     "publisher": "MoSPI, National Accounts Statistics",
     "source": "https://api.mospi.gov.in (nas/getNASData, indicator_code=25)",
     "accessed": fetched,
     "terms": "published statistics; cite MoSPI",
     "in_project_repo": "yes"},
    {"file": "PLFS_DIR/perv1.csv and Data_LayoutPLFS_2023-24.xlsx",
     "dataset": "PLFS July 2023-June 2024, person file, first visit",
     "publisher": "NSO, MoSPI",
     "source": "https://microdata.gov.in (registration required)",
     "accessed": str(plfs_date.date()),
     "terms": "no redistribution; no attempt to identify respondents; cite NSO",
     "in_project_repo": "NO"},
])
data_log.to_csv(PROJECT / "data_log.csv", index=False)
print(data_log[["dataset", "accessed", "in_project_repo"]].to_string(index=False))
print("Terms:", *data_log["terms"], sep="\n  ")

## 30.4 A README that a stranger can follow

In [ ]:
readme = f"""# Do richer Indian states have more salaried jobs?

Dissertation chapter, 2023-24 data. Every table and figure in `output/`
is rebuilt from `data/raw/` and the PLFS files by one command.

## Data
See `data_log.csv`. The PLFS files are NOT included: register at
https://microdata.gov.in, download PLFS 2023-24 (CSV and data layout) and
put them in `data/plfs/` or set the environment variable PLFS_DIR.

## Rebuild
    python -m venv .venv
    .venv\\Scripts\\activate            (Mac/Linux: source .venv/bin/activate)
    pip install -r requirements.txt
    python code/run_all.py

## Layout
code/salaried.py   cleaning and analysis functions
code/run_all.py    rebuilds data/clean/ and output/ from scratch
"""
(PROJECT / "README.md").write_text(readme, encoding="utf-8")
print(readme[:200])

## 30.5 Cleaning in functions, in a module

In [ ]:
MODULE = r'''"""Do richer states have more salaried jobs?  Project functions.

All cleaning lives here; the notebook and run_all.py only call these
functions. PLFS unit records are read into memory, never written to disk.
"""
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

from indiadata import read_plfs

# gedu_lvl codes (PLFS Schedule 10.4, see Chapter 18) in five levels
EDU = {1: "None", 2: "None", 3: "None", 4: "None", 5: "Primary",
       6: "Primary", 7: "Middle/secondary", 8: "Middle/secondary",
       10: "Higher sec./diploma", 11: "Higher sec./diploma",
       12: "Graduate+", 13: "Graduate+"}
EDU_ORDER = ["None", "Primary", "Middle/secondary", "Higher sec./diploma",
             "Graduate+"]
CROSSWALK = {"Tamilnadu": "Tamil Nadu", "Puduchery": "Puducherry",
             "Andaman & N. Island": "Andaman & Nicobar Islands"}  # Chapter 7
UNION_TERRITORIES = ["Andaman & Nicobar Islands", "Chandigarh", "Delhi",
                     "Jammu & Kashmir", "Puducherry"]


def state_names(plfs_dir):
    """PLFS state codes with NAS-style names, from the layout workbook."""
    layout = next(Path(plfs_dir).glob("Data_Layout*.xlsx"))
    st = pd.read_excel(layout, sheet_name="State code", header=1)
    st.columns = ["st", "plfs_name"]
    st = st.dropna().astype({"st": int})
    st["state"] = st["plfs_name"].str.strip().replace(CROSSWALK)
    assert st["st"].is_unique
    return st[["st", "state"]]


def state_income(raw_csv, year="2023-24"):
    """Per capita NSDP at current prices (Rs) in one fiscal year."""
    raw = pd.read_csv(raw_csv, dtype=str)
    inc = raw[raw["year"] == year].copy()
    inc["pc_nsdp"] = pd.to_numeric(inc["current_price"], errors="raise")
    assert inc["unit"].eq("₹").all() and inc["state"].is_unique
    inc["log_pc"] = np.log(inc["pc_nsdp"] / 1e5)          # log of Rs lakh
    return inc[["state", "pc_nsdp", "log_pc"]].reset_index(drop=True)


def load_workers(plfs_dir):
    """Workers aged 15-59 by usual principal status: one row per person."""
    p = read_plfs("perv1", folder=plfs_dir,
                  columns=["sex", "age", "gedu_lvl", "pas"])
    w = p[p["pas"].between(11, 51) & p["age"].between(15, 59)
          & p["sex"].isin([1, 2])]
    d = pd.DataFrame({
        "salaried": (w["pas"] == 31).astype(float),
        "female": (w["sex"] == 2).astype(float),
        "urban": (w["sec"] == 2).astype(float),
        "age": w["age"].astype(float),
        "edu": pd.Categorical(w["gedu_lvl"].map(EDU), categories=EDU_ORDER),
        "st": w["st"].astype(int),
        "weight": w["weight"].astype(float)})
    assert d["edu"].notna().all(), "unmapped education code"
    return d.reset_index(drop=True)


def add_state(workers, names, income):
    """Attach state names and income; keep states with an income figure."""
    d = workers.merge(names, on="st", how="left", validate="many_to_one")
    d = d.merge(income, on="state", how="inner", validate="many_to_one")
    d["w_norm"] = d["weight"] / d["weight"].mean()       # weights averaging 1
    return d


def state_table(d):
    """Weighted state shares (%) and sample sizes: aggregates only."""
    def one(g):
        w = g["weight"]
        return pd.Series({
            "salaried_pct": 100 * np.average(g["salaried"], weights=w),
            "graduate_pct": 100 * np.average(g["edu"] == "Graduate+", weights=w),
            "urban_pct": 100 * np.average(g["urban"], weights=w),
            "workers_mn": w.sum() / 1e6, "sample": len(g),
            "pc_nsdp": g["pc_nsdp"].iloc[0]})
    cols = ["salaried", "edu", "urban", "weight", "pc_nsdp"]
    return d.groupby("state")[cols].apply(one).sort_values("pc_nsdp")


CONTROLS = "female + urban + age + I(age**2 / 100)"
FORMULAS = {"(1)": "salaried ~ log_pc",
            "(2)": "salaried ~ log_pc + " + CONTROLS,
            "(3)": "salaried ~ log_pc + " + CONTROLS + " + C(edu)",
            "(4)": "salaried ~ log_pc + " + CONTROLS + " + C(edu)"}


def fit_models(d):
    """Weighted linear probability models, SEs clustered by state.
    Model (4) repeats (3) without the union territories."""
    out = {}
    for name, formula in FORMULAS.items():
        data = d[~d["state"].isin(UNION_TERRITORIES)] if name == "(4)" else d
        res = smf.wls(formula, data=data, weights=data["w_norm"]).fit(
            cov_type="cluster", cov_kwds={"groups": data["st"]})
        res.n_clusters = data["st"].nunique()
        out[name] = res
    return out


LABELS = {"log_pc": "Log per capita NSDP", "female": "Female",
          "urban": "Urban", "age": "Age", "I(age ** 2 / 100)": "Age squared / 100",
          **{f"C(edu)[T.{e}]": f"Education: {e}" for e in EDU_ORDER[1:]},
          "Intercept": "Constant"}


def regression_table(models):
    """Side-by-side table of all models, built with summary_col."""
    table = summary_col(
        list(models.values()), model_names=list(models), stars=True,
        float_format="%.3f", regressor_order=list(LABELS), drop_omitted=True,
        info_dict={"Observations": lambda r: f"{int(r.nobs):,}",
                   "States (clusters)": lambda r: str(r.n_clusters)})
    t = table.tables[0]
    t.index = [LABELS.get(i, i) for i in t.index]
    return table


def to_markdown(df):
    """A Markdown table without extra packages."""
    head = "| | " + " | ".join(map(str, df.columns)) + " |"
    rule = "|---" * (len(df.columns) + 1) + "|"
    rows = ["| " + str(i) + " | " + " | ".join(map(str, r)) + " |"
            for i, r in zip(df.index, df.to_numpy())]
    return "\n".join([head, rule, *rows])


STYLE = {"font.size": 9, "axes.titlesize": 9.5, "axes.labelsize": 9,
         "xtick.labelsize": 8, "ytick.labelsize": 8,
         "axes.spines.top": False, "axes.spines.right": False,
         "pdf.fonttype": 42, "figure.dpi": 100, "figure.facecolor": "white"}
OFFSETS = {"Bihar": (5, 3), "Uttar Pradesh": (6, 4), "Kerala": (4, 3),
           "Maharashtra": (6, -10), "Delhi": (-26, 4), "Goa": (4, -9),
           "Sikkim": (-14, 6)}


def figure_states(tab):
    """Figure 1: salaried and graduate shares against state income."""
    from matplotlib.ticker import NullFormatter
    with plt.rc_context(STYLE):
        fig, axes = plt.subplots(1, 2, figsize=(6.3, 2.9), sharex=True,
                                 layout="constrained")
        x = tab["pc_nsdp"] / 1e3
        for ax, col, title in zip(axes, ["salaried_pct", "graduate_pct"],
                                  ["(a) Salaried, % of workers",
                                   "(b) Graduates, % of workers"]):
            ax.scatter(x, tab[col], s=8 + 2.5 * tab["workers_mn"],
                       color="#1f4e79", alpha=0.6, edgecolor="white", lw=0.5)
            b = np.polyfit(np.log(x), tab[col], 1, w=np.sqrt(tab["workers_mn"]))
            grid = np.linspace(np.log(x.min()), np.log(x.max()), 50)
            ax.plot(np.exp(grid), np.polyval(b, grid), color="#c55a11", lw=1)
            for st, off in OFFSETS.items():
                ax.annotate(st, (x[st], tab.loc[st, col]), xytext=off,
                            textcoords="offset points", fontsize=7)
            ax.set_xscale("log")
            ax.set_xticks([60, 100, 200, 400], labels=["60", "100", "200", "400"])
            ax.xaxis.set_minor_formatter(NullFormatter())
            ax.set_title(title, loc="left")
        fig.supxlabel("Per capita NSDP 2023-24, Rs thousand (log scale)",
                      fontsize=9)
    return fig


def save(fig, stem):
    """Save as vector PDF and 300-dpi PNG; no timestamps, so reruns match."""
    fig.savefig(Path(stem).with_suffix(".pdf"), bbox_inches="tight",
                metadata={"CreationDate": None})
    fig.savefig(Path(stem).with_suffix(".png"), bbox_inches="tight", dpi=300)
'''
(PROJECT / "code" / "salaried.py").write_text(MODULE, encoding="utf-8")

sys.path.insert(0, str(PROJECT / "code"))       # let Python find the module
import salaried as s
s = importlib.reload(s)                         # pick up edits on a rerun
[name for name in dir(s) if not name.startswith("_") and callable(getattr(s, name))
 and getattr(s, name).__module__ == "salaried"]

## 30.6 First, reproduce a published number

In [ ]:
from indiadata import read_plfs, labour_status, labour_rates

p = read_plfs("perv1", folder=PLFS_DIR, columns=["age", "pas", "sas"])
p["status"] = labour_status(p["pas"], p["sas"])
check = pd.DataFrame({"ours": labour_rates(p[p["age"] >= 15]).round(1),
                      "MoSPI": [60.1, 58.2, 3.2]})
del p
check

In [ ]:
names = s.state_names(PLFS_DIR)
income = s.state_income(raw_income)
income.to_csv(PROJECT / "data/clean/state_income_2023_24.csv", index=False)
workers = s.load_workers(PLFS_DIR)
d = s.add_state(workers, names, income)

dropped = sorted(set(names["state"]) - set(income["state"]))
print(f"Workers aged 15-59 (usual principal status): {len(workers):,}")
print(f"... in the {d['state'].nunique()} states with per capita NSDP: {len(d):,}")
print("No per capita NSDP (code 25):", dropped)
del workers

## 30.7 The descriptive picture

In [ ]:
tab = s.state_table(d)
tab.round(1).astype({"sample": int, "pc_nsdp": int}).iloc[np.r_[0:3, -3:0]]

In [ ]:
r = np.corrcoef(np.log(tab["pc_nsdp"]), tab["salaried_pct"])[0, 1]
print(f"States: {len(tab)}; correlation of salaried share with log income: {r:.2f}")
print(f"Salaried share: {tab['salaried_pct'].min():.1f}% ({tab['salaried_pct'].idxmin()}) "
      f"to {tab['salaried_pct'].max():.1f}% ({tab['salaried_pct'].idxmax()})")
tab.round(2).to_csv(PROJECT / "output/tables/table1_state_shares.csv")

In [ ]:
fig = s.figure_states(tab)
s.save(fig, PROJECT / "output/figures/figure1_states")
plt.show()

## 30.8 Regressions and a publication table

In [ ]:
models = s.fit_models(d)
table = s.regression_table(models)
print(table)

In [ ]:
b = pd.Series({k: m.params["log_pc"] for k, m in models.items()})
print("Coefficient on log per capita NSDP:", b.round(3).to_dict())
print(f"Doubling income, model (1): {100 * b['(1)'] * np.log(2):.1f} points more "
      f"likely to be salaried; model (3): {100 * b['(3)'] * np.log(2):.1f} points")
print(f"Share of the raw gradient left after controls: {b['(3)'] / b['(1)']:.0%}")

## 30.9 Exporting the table: LaTeX, Markdown, Word

In [ ]:
out = PROJECT / "output/tables"
t = table.tables[0]                                   # a pandas DataFrame
t.to_csv(out / "table2_regressions.csv")              # opens in Excel and Word
(out / "table2_regressions.tex").write_text(table.as_latex(), encoding="utf-8")
(out / "table2_regressions.md").write_text(s.to_markdown(t), encoding="utf-8")
print(s.to_markdown(t.iloc[:6]))

## 30.10 Is the result robust?

In [ ]:
ci = pd.DataFrame({k: m.conf_int().loc["log_pc"] for k, m in models.items()}).T
ci.columns = ["low", "high"]
labels = ["(1) Income only", "(2) + sex, urban, age", "(3) + education",
          "(4) = (3) without UTs"]

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.errorbar(b.values, range(len(b)), xerr=[b - ci["low"], ci["high"] - b],
            fmt="o", color="#1f4e79", ecolor="#9db7d5", elinewidth=3, capsize=0)
ax.axvline(0, color="black", lw=0.8)
ax.set_yticks(range(len(b)), labels=labels)
ax.invert_yaxis()
ax.set_xlim(left=0)
ax.set_xlabel("Coefficient on log per capita NSDP (change in probability)")
ax.set_title("The income gradient shrinks with controls but does not vanish",
             loc="left", fontsize=12)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.06, "95% confidence intervals, SEs clustered by state. "
         "Source: author's calculations from PLFS 2023-24 and MoSPI NAS.",
         fontsize=8, color="grey")
plt.show()

In [ ]:
d["income_group"] = d["state"].map(pd.qcut(tab["pc_nsdp"], 3,
                                           labels=["Poorest 10", "Middle 10",
                                                   "Richest 10"]))
within = (d.groupby(["edu", "income_group"], observed=True)
           .apply(lambda g: pd.Series({
               "salaried_pct": 100 * np.average(g["salaried"], weights=g["weight"]),
               "sample": len(g)}))
           .unstack("income_group"))
within["salaried_pct"].round(1)

In [ ]:
shares = within["salaried_pct"]
fig, ax = plt.subplots(figsize=(9, 4))
shares.plot.bar(ax=ax, color=["#9db7d5", "#1f4e79", "#c55a11"], width=0.8, rot=0)
ax.set_xticks(range(len(shares)),
              labels=[e.replace("/", "/\n") for e in shares.index])
ax.set_ylabel("Salaried, % of workers")
ax.set_xlabel("")
ax.set_title("Within every education level, workers in richer states are "
             "more often salaried", loc="left", fontsize=12)
ax.legend(title="States grouped by per capita NSDP", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.08, "Workers aged 15-59, usual principal status, weighted. "
         "Source: author's calculations from PLFS 2023-24 and MoSPI NAS.",
         fontsize=8, color="grey")
plt.show()

In [ ]:
gap = shares["Richest 10"] - shares["Poorest 10"]
print("Richest minus poorest group, percentage points:")
print(gap.round(1).to_string())
print(f"Smallest cell: {int(within['sample'].min().min()):,} workers")

## 30.11 Figures at publication quality

In [ ]:
for f in sorted((PROJECT / "output/figures").iterdir()):
    print(f"{f.name:22s} {f.stat().st_size / 1e3:6.0f} kB")

## 30.12 One script to rebuild everything

In [ ]:
RUN_ALL = r'''"""Rebuild every table and figure from the raw data.

Usage, from the project folder:  python code/run_all.py
Set the environment variable PLFS_DIR if the PLFS files are not in data/plfs.
"""
import os
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "code"))
import salaried as s                                        # noqa: E402

PLFS_DIR = Path(os.environ.get("PLFS_DIR", ROOT / "data" / "plfs"))
RAW, CLEAN = ROOT / "data" / "raw", ROOT / "data" / "clean"
TABLES, FIGS = ROOT / "output" / "tables", ROOT / "output" / "figures"
for folder in (CLEAN, TABLES, FIGS):
    folder.mkdir(parents=True, exist_ok=True)

# 1. Clean. Public aggregates are saved; PLFS records stay in memory.
income = s.state_income(RAW / "mospi_nas_pc_nsdp_code25.csv")
income.to_csv(CLEAN / "state_income_2023_24.csv", index=False)
d = s.add_state(s.load_workers(PLFS_DIR), s.state_names(PLFS_DIR), income)

# 2. Analyse and write every output.
tab = s.state_table(d)
tab.round(2).to_csv(TABLES / "table1_state_shares.csv")
table = s.regression_table(s.fit_models(d))
table.tables[0].to_csv(TABLES / "table2_regressions.csv")
(TABLES / "table2_regressions.tex").write_text(table.as_latex(), encoding="utf-8")
(TABLES / "table2_regressions.md").write_text(s.to_markdown(table.tables[0]),
                                              encoding="utf-8")
s.save(s.figure_states(tab), FIGS / "figure1_states")
print(f"done: {len(d):,} workers, {d['state'].nunique()} states")
'''
(PROJECT / "code" / "run_all.py").write_text(RUN_ALL, encoding="utf-8")
print(f"code/run_all.py written: {len(RUN_ALL.splitlines())} lines")

In [ ]:
import gc
import hashlib
import os
import subprocess


def fingerprints(folder: Path) -> pd.Series:
    """SHA-256 of every file below a folder (first 12 hex digits)."""
    files = sorted(f for f in folder.rglob("*") if f.is_file())
    return pd.Series({str(f.relative_to(folder)):
                      hashlib.sha256(f.read_bytes()).hexdigest()[:12]
                      for f in files})


before = pd.concat([fingerprints(PROJECT / "output"),
                    fingerprints(PROJECT / "data/clean")])
del d, models, table, fig                      # free memory for the new process
plt.close("all")
gc.collect()

shutil.rmtree(PROJECT / "output")              # delete what code made ...
shutil.rmtree(PROJECT / "data/clean")
env = {**os.environ, "PLFS_DIR": str(PLFS_DIR), "MPLBACKEND": "Agg",
       "OPENBLAS_NUM_THREADS": "1", "OMP_NUM_THREADS": "1"}   # low-RAM laptops
run = subprocess.run([sys.executable, "code/run_all.py"], cwd=PROJECT, env=env,
                     capture_output=True, text=True, timeout=600)
print("exit code:", run.returncode, "|", run.stdout.strip(), run.stderr[-500:])

after = pd.concat([fingerprints(PROJECT / "output"),
                   fingerprints(PROJECT / "data/clean")])
test = pd.DataFrame({"before": before, "after": after})
test["same"] = test["before"] == test["after"]
test

## 30.13 Environments and `requirements.txt`

In [ ]:
from importlib.metadata import version

PACKAGES = ["pandas", "numpy", "statsmodels", "matplotlib", "openpyxl"]
pins = {p: version(p) for p in PACKAGES}
print("Python", sys.version.split()[0])
requirements = "\n".join(f"{p}=={v}" for p, v in pins.items()) + (
    "\nindiadata @ git+https://github.com/slxnjdhv/python-for-indian-economics.git\n")
(PROJECT / "requirements.txt").write_text(requirements, encoding="utf-8")
print(requirements)

## 30.14 Version control with Git

In [ ]:
gitignore = """# PLFS microdata: MoSPI terms forbid redistribution
data/plfs/
*.zip
# rebuilt by code/run_all.py
output/
data/clean/
# Python and Jupyter clutter
.venv/
__pycache__/
.ipynb_checkpoints/
"""
(PROJECT / ".gitignore").write_text(gitignore, encoding="utf-8")
tree(PROJECT)

## 30.15 Citing data and software

In [ ]:
from importlib.metadata import version as v

print(f"Analysis in Python {sys.version.split()[0]} with pandas {v('pandas')}, "
      f"NumPy {v('numpy')}, statsmodels {v('statsmodels')} and "
      f"matplotlib {v('matplotlib')}.")

## 30.16 Ethics, licences and re-identification

In [ ]:
t1 = pd.read_csv(PROJECT / "output/tables/table1_state_shares.csv",
                 index_col="state")
print(f"Smallest state sample in Table 1: {int(t1['sample'].min())} workers "
      f"({t1['sample'].idxmin()})")
print("States with fewer than 1,000 sampled workers:",
      t1.index[t1["sample"] < 1000].tolist())

## 30.17 Using AI assistants responsibly

## 30.18 Answering the question

## Key ideas

## Exercises

## Data citation and terms of use

## Further reading